# IPL Auction Intelligence: Identifying Reliable Players

IPL franchises invest heavily in player auctions, but the most expensive player is not always the most valuable. To make smarter auction decisions, we need to identify players who consistently contribute to their team's success.

## Key Qualities of a Valuable Player

Consistency – Performs well across multiple seasons.

Match Impact – Contributes to winning matches.

Pressure Performance – Delivers in crucial matches such as playoffs and finals.

Reliability – Maintains performance over a long period.

Efficiency – Makes the most of the opportunities available.


### This analysis aims to identify:
*The most reliable batters.

*The most reliable bowlers.

*Players who perform under pressure.

*Players who provide the best return on auction investment.


In [ ]:
#Importing Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
#Loading Dataset
ipl_dataset = pd.read_csv(
    "data/ipl_dataset.csv",
    low_memory=False
)

In [ ]:
ipl_dataset.head()

## Data Loading

The IPL ball-by-ball dataset is loaded using Pandas. The dataset contains detailed information about every delivery bowled in IPL history, including batting performance, bowling performance, match outcomes, venues, and player statistics.

This dataset will serve as the foundation for evaluating player reliability and identifying valuable auction investments.

In [ ]:
print("Dataset Shape:", ipl_dataset.shape)
print("Number of Rows:", ipl_dataset.shape[0])
print("Number of Columns:", ipl_dataset.shape[1])

## Dataset Dimensions

Understanding the size of the dataset is the first step in any analysis.
The number of rows represents the total ball-by-ball records, while the number of columns represents the available features that can be used for analysis.

print("Columns in Dataset:\n")

for col in ipl_dataset.columns:
    print(col)

In [ ]:
ipl_dataset.info()

In [ ]:
ipl_dataset.dtypes

## Data Types

Examining data types helps identify whether each column contains numerical, categorical, or date-related information. This step is important because incorrect data types can affect calculations and visualizations later in the analysis.

In [ ]:
#Summary
print(f"Total Records: {ipl_dataset.shape[0]}")
print(f"Total Features: {ipl_dataset.shape[1]}")

In [ ]:
#Count Missing Values
missing_values = ipl_dataset.isnull().sum()

missing_values[missing_values > 0].sort_values(ascending=False)

In [ ]:
#Missing value percentage
missing_percentage = (
    ipl_dataset.isnull().sum() /
    len(ipl_dataset)
) * 100

missing_df = pd.DataFrame({
    'Missing Values': ipl_dataset.isnull().sum(),
    'Percentage': missing_percentage
})

missing_df = missing_df[
    missing_df['Missing Values'] > 0
].sort_values(
    by='Percentage',
    ascending=False
)

missing_df

## Missing Value Analysis

Missing values can affect the quality of analysis and lead to misleading conclusions.

The missing value analysis helps identify:

- Columns that are largely incomplete.
- Features that may not be useful for player evaluation.
- Data quality issues that need to be addressed before analysis.

Columns with extremely high missing percentages may be removed if they do not contribute significantly to the business objective.

In [ ]:
# -----------------------------
# Remove columns with extremely high missing values
# and low relevance to player, team, and venue analysis
# -----------------------------

drop_cols = [
    'power_surge_start',
    'review_batter',
    'team_reviewed',
    'review_decision',
    'umpire',
    'method',
    'superover_winner'
]

ipl_dataset.drop(columns=drop_cols, inplace=True)

# Verify removal
print("Updated Dataset Shape:", ipl_dataset.shape)
print("\nRemaining Columns:", len(ipl_dataset.columns))

## Missing Value Treatment

Several columns contained more than 95% missing values and were not directly relevant to player evaluation, team performance, or venue analysis. These columns were removed to improve data quality and reduce noise.

The following columns were retained despite having missing values:

- wicket_kind
- player_out
- fielders
- extra_type
- result_type
- runs_target

Missing values in these columns are expected because they correspond to specific match events that do not occur on every ball.

In [ ]:
# Check duplicate records
print("Duplicate Rows:", ipl_dataset.duplicated().sum())

## Duplicate Record Analysis

Duplicate records can lead to inaccurate calculations and biased insights. Therefore, the dataset was checked for duplicate entries before analysis.

The dataset contained no duplicate rows, indicating good data quality and ensuring that each ball-by-ball event is uniquely recorded.

In [ ]:
sorted(ipl_dataset['batting_team'].dropna().unique())

In [ ]:
team_mapping = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Rising Pune Supergiant': 'Rising Pune Supergiants',
    'Royal Challengers Bengaluru': 'Royal Challengers Bangalore'
}

team_cols = [
    'batting_team',
    'bowling_team',
    'toss_winner',
    'match_won_by'
]

ipl_dataset[team_cols] = (
    ipl_dataset[team_cols]
    .replace(team_mapping)
)

In [ ]:
'Unnamed: 0' in ipl_dataset.columns

In [ ]:
ipl_dataset.drop(columns=['Unnamed: 0'], inplace=True)

In [ ]:
#Convert Date Column
ipl_dataset['date'] = pd.to_datetime(
    ipl_dataset['date'],
    errors='coerce'
)

In [ ]:
ipl_dataset['date'].dtype

In [ ]:
numeric_cols = [
    'over',
    'ball',
    'runs_batter',
    'runs_extras',
    'runs_total',
    'runs_bowler',
    'team_runs',
    'team_balls',
    'team_wicket',
    'balls_faced',
    'bat_pos'
]

for col in numeric_cols:
    if col in ipl_dataset.columns:
        ipl_dataset[col] = pd.to_numeric(
            ipl_dataset[col],
            errors='coerce'
        )

## Data Type Standardization

To ensure accurate analysis, important columns were converted to appropriate data types. The date column was converted to datetime format, while numerical features were validated and converted where necessary.

This step improves consistency and enables time-based and statistical analyses.

In [ ]:
matches = ipl_dataset.drop_duplicates(
    subset='match_id'
).copy()

In [ ]:
sorted(ipl_dataset['batting_team'].dropna().unique())

In [ ]:
sorted(ipl_dataset['bowling_team'].dropna().unique())

In [ ]:
ipl_dataset['batting_team'].nunique()

In [ ]:
ipl_dataset['batting_team'].value_counts()

In [ ]:
matches = ipl_dataset.drop_duplicates(
    subset='match_id'
).copy()

In [ ]:
total_matches = matches['match_id'].nunique()

total_players = pd.concat([
    ipl_dataset['batter'],
    ipl_dataset['bowler']
]).nunique()

total_teams = pd.concat([
    matches['batting_team'],
    matches['bowling_team']
]).nunique()

total_venues = matches['venue'].nunique()

total_seasons = matches['season'].nunique()

In [ ]:
print("Total Matches :", total_matches)
print("Total Players :", total_players)
print("Total Teams   :", total_teams)
print("Total Venues  :", total_venues)
print("Total Seasons :", total_seasons)

## Dataset Overview

Before evaluating players and teams, it is important to understand the scale of the dataset.

The IPL dataset contains ball-by-ball information across multiple seasons, teams, venues, and players. This provides a strong foundation for identifying reliable players and making data-driven auction decisions.

In [ ]:
#Team Analysis 1: Most Successful Teams
team_wins = (
    matches['match_won_by']
    .value_counts()
    .head(10)
)

team_wins

In [ ]:
plt.figure(figsize=(10,5))

bars = plt.bar(
    team_wins.index,
    team_wins.values
)

plt.title("Most Successful IPL Teams")
plt.xlabel("Team")
plt.ylabel("Matches Won")

plt.xticks(rotation=45)
plt.grid(axis='y')

plt.bar_label(bars)

plt.show()

## Most Successful Teams

The number of matches won provides a direct measure of a team's historical success.

Teams with consistently high win counts demonstrate strong squad composition, effective leadership, and long-term performance across seasons.

In [ ]:
#Mumbai Indians and Chennai Super Kings have been the most successful franchises in IPL history, 
# suggesting that long-term squad stability and smart player investments contribute significantly
# to sustained success.

Team Analysis 2: Win Percentage

In [ ]:
#Step 1: Matches Played
matches_played = pd.concat([
    matches['batting_team'],
    matches['bowling_team']
]).value_counts()

In [ ]:
#Step 2: Matches Won
matches_won = (
    matches['match_won_by']
    .replace('Unknown', np.nan)
    .dropna()
    .value_counts()
)

In [ ]:
#Step 3: Create DataFrame
team_stats = pd.DataFrame({
    'Matches_Played': matches_played,
    'Matches_Won': matches_won
}).fillna(0)

In [ ]:
#Step 4: Win Percentage
team_stats['Win_Percentage'] = (
    team_stats['Matches_Won']
    /
    team_stats['Matches_Played']
) * 100


In [ ]:
#Step 5: Sort
team_stats = team_stats.sort_values(
    'Win_Percentage',
    ascending=False
)
team_stats.head(10)

In [ ]:
top_win_pct = team_stats.head(10)

plt.figure(figsize=(10,5))

bars = plt.bar(
    top_win_pct.index,
    top_win_pct['Win_Percentage']
)

plt.title("IPL Teams by Win Percentage")
plt.xlabel("Team")
plt.ylabel("Win Percentage (%)")

plt.xticks(rotation=45)
plt.grid(axis='y')

plt.bar_label(bars, fmt='%.1f')

plt.show()

Gujarat Titans have achieved the highest win percentage despite being one of the newest franchises, indicating exceptional team construction and strategy.


CSK combines longevity with a high win percentage, making them the benchmark for sustained IPL success.

Mumbai Indians have the highest number of wins while maintaining an excellent win rate, highlighting the value of retaining a strong core squad.

Despite having star players and a high number of wins, RCB's win percentage remains below 50%, suggesting that star power alone does not guarantee long-term success.

## Team Analysis 3: Does Winning the Toss Matter?

In [ ]:
#Step 1: Calculate Toss Impact
toss_advantage = (
    matches['toss_winner']
    ==
    matches['match_won_by']
)

toss_win_percentage = (
    toss_advantage.mean()
) * 100

print(f"Toss Winner Won Match: {toss_win_percentage:.2f}%")

In [ ]:
#Step 2: Count Wins vs Losses After Toss
toss_results = toss_advantage.value_counts()

toss_results

In [ ]:
#Step 3: Visualize
plt.figure(figsize=(6,6))

plt.pie(
    toss_results,
    labels=['Toss Winner Won', 'Toss Winner Lost'],
    autopct='%1.1f%%'
)

plt.title('Impact of Toss on Match Outcome')

plt.show()

## Toss Impact Analysis

Winning the toss is often considered a major advantage in T20 cricket. This analysis evaluates whether the team winning the toss also wins the match.

By comparing toss winners with match winners, we can determine the actual influence of the toss on match outcomes.

## Venue Analysis 1: Most Frequently Used Venues

In [ ]:
top_venues = matches['venue'].value_counts().head(10)

top_venues

In [ ]:
plt.figure(figsize=(10,5))

bars = plt.bar(
    top_venues.index,
    top_venues.values
)

plt.title("Most Frequently Used IPL Venues")
plt.xlabel("Venue")
plt.ylabel("Matches Hosted")

plt.xticks(rotation=90)

plt.bar_label(bars)

plt.show()

## Venue Analysis 2: Highest Scoring Venues

In [ ]:
matches[['venue','team_runs']].head()

In [ ]:
# Issue 1: team_runs is not usable
# Issue 2: Venue Names Need Cleaning
venue_mapping = {
    'Wankhede Stadium, Mumbai': 'Wankhede Stadium',
    'MA Chidambaram Stadium, Chepauk, Chennai': 'MA Chidambaram Stadium, Chepauk'
}

ipl_dataset['venue'] = (
    ipl_dataset['venue']
    .replace(venue_mapping)
)

matches['venue'] = (
    matches['venue']
    .replace(venue_mapping)
)

## Batter Analysis 1: Top Run Scorers

In [ ]:
top_runs = (
    ipl_dataset.groupby('batter')['runs_batter']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_runs

In [ ]:
plt.figure(figsize=(10,5))

bars = plt.bar(
    top_runs.index,
    top_runs.values
)

plt.title("Top Run Scorers in IPL")
plt.xlabel("Batter")
plt.ylabel("Runs")

plt.xticks(rotation=45)

plt.bar_label(bars)

plt.show()

Players such as Virat Kohli, Rohit Sharma, and Shikhar Dhawan have consistently accumulated runs across multiple seasons, making them dependable batting assets.

## Batter Analysis 2: Strike Rate

Runs alone are not enough.

A batter scoring 500 runs at SR 150 is often more valuable in T20 than one scoring 500 at SR 120.

In [ ]:
batter_stats = (
    ipl_dataset.groupby('batter')
    .agg({
        'runs_batter':'sum',
        'balls_faced':'sum'
    })
)

batter_stats = batter_stats[
    batter_stats['balls_faced'] >= 500
]

batter_stats['Strike_Rate'] = (
    batter_stats['runs_batter']
    /
    batter_stats['balls_faced']
) * 100

top_sr = (
    batter_stats['Strike_Rate']
    .sort_values(ascending=False)
    .head(10)
)

top_sr

In [ ]:
plt.figure(figsize=(10,5))

bars = plt.bar(
    top_sr.index,
    top_sr.values
)

plt.title("Highest Strike Rate Batters")
plt.ylabel("Strike Rate")

plt.xticks(rotation=45)

plt.bar_label(bars, fmt='%.1f')

plt.show()

High strike-rate batters increase scoring efficiency and can significantly influence match outcomes.

## Batter Analysis 3: Player of the Match Awards


In [ ]:
potm = (
    matches['player_of_match']
    .value_counts()
    .head(10)
)

potm

In [ ]:
plt.figure(figsize=(10,5))

bars = plt.bar(
    potm.index,
    potm.values
)

plt.title("Most Player of the Match Awards")
plt.ylabel("Awards")

plt.xticks(rotation=45)

plt.bar_label(bars)

plt.show()

Player of the Match awards indicate a batter's ability to influence match outcomes rather than simply accumulate runs.

## Batter Analysis 4: Death Over Specialists

Auction teams love finishers.

In [ ]:
death_overs = ipl_dataset[
    ipl_dataset['over'] >= 16
]

death_runs = (
    death_overs.groupby('batter')['runs_batter']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

death_runs

In [ ]:
plt.figure(figsize=(10,5))

bars = plt.bar(
    death_runs.index,
    death_runs.values
)

plt.title("Top Death Over Run Scorers")
plt.ylabel("Runs")

plt.xticks(rotation=45)

plt.bar_label(bars)

plt.show()

Players who score heavily in death overs provide exceptional value because they perform during the most impactful phase of a T20 innings.

## Batter Analysis 5: Consistency

How many seasons has a batter played?


In [ ]:
batter_seasons = (
    ipl_dataset.groupby('batter')['season']
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)

batter_seasons

Longevity is an indicator of reliability and adaptability across changing teams, venues, and conditions.

## Batter Reliability Index (BRI)


In [ ]:
#Step 1: Create Total Runs Data
runs = (
    ipl_dataset.groupby('batter')['runs_batter']
    .sum()
)

In [ ]:
#Step 2: Create Strike Rate Data
strike_rate = (
    ipl_dataset.groupby('batter')
    .agg({
        'runs_batter':'sum',
        'balls_faced':'sum'
    })
)

strike_rate = strike_rate[
    strike_rate['balls_faced'] >= 500
]

strike_rate['Strike_Rate'] = (
    strike_rate['runs_batter']
    /
    strike_rate['balls_faced']
) * 100

strike_rate = strike_rate['Strike_Rate']

In [ ]:
#Step 3: POTM Count
potm_count = (
    matches['player_of_match']
    .value_counts()
)

In [ ]:
#Step 4: Death Over Runs
death_runs = (
    ipl_dataset[
        ipl_dataset['over'] >= 16
    ]
    .groupby('batter')['runs_batter']
    .sum()
)

In [ ]:
#Step 5: Seasons Played
seasons = (
    ipl_dataset.groupby('batter')['season']
    .nunique()
)

In [ ]:
#Combine Everything
bri = pd.DataFrame({
    'Runs': runs,
    'Strike_Rate': strike_rate,
    'POTM': potm_count,
    'Death_Runs': death_runs,
    'Seasons': seasons
})

In [ ]:
#Fill Missing Values
#Some players won't have POTM awards.
bri = bri.fillna(0)

In [ ]:
#Normalize Scores
bri['Runs_Norm'] = (
    bri['Runs']
    /
    bri['Runs'].max()
)

bri['SR_Norm'] = (
    bri['Strike_Rate']
    /
    bri['Strike_Rate'].max()
)

bri['POTM_Norm'] = (
    bri['POTM']
    /
    bri['POTM'].max()
)

bri['Death_Norm'] = (
    bri['Death_Runs']
    /
    bri['Death_Runs'].max()
)

bri['Season_Norm'] = (
    bri['Seasons']
    /
    bri['Seasons'].max()
)

## Create Reliability Score

Weights:

Runs = 35%
Strike Rate = 20%
POTM = 20%
Death Runs = 15%
Seasons = 10%

In [ ]:
bri['BRI'] = (
    0.35 * bri['Runs_Norm']
    + 0.20 * bri['SR_Norm']
    + 0.20 * bri['POTM_Norm']
    + 0.15 * bri['Death_Norm']
    + 0.10 * bri['Season_Norm']
)

In [ ]:
#Top Auction Targets
top_batters = (
    bri.sort_values(
        'BRI',
        ascending=False
    )
    .head(10)
)

top_batters

In [ ]:
plt.figure(figsize=(10,5))

bars = plt.bar(
    top_batters.index,
    top_batters['BRI']
)

plt.title("Top Batters by Reliability Index")
plt.ylabel("BRI Score")

plt.xticks(rotation=45)

plt.bar_label(bars, fmt='%.2f')

plt.show()

Instead of selecting batters solely based on total runs, the Batter Reliability Index evaluates consistency, scoring efficiency, match-winning impact, finishing ability, and longevity. Players with the highest BRI represent the safest and most valuable auction investments.

## Bowler Analysis 1: Top Wicket Takers

In [ ]:
top_wickets = (
    ipl_dataset[
        ipl_dataset['player_out'].notna()
    ]
    .groupby('bowler')
    .size()
    .sort_values(ascending=False)
    .head(10)
)

top_wickets

In [ ]:
plt.figure(figsize=(10,5))

bars = plt.bar(
    top_wickets.index,
    top_wickets.values
)

plt.title("Top Wicket Takers in IPL")
plt.ylabel("Wickets")

plt.xticks(rotation=45)

plt.bar_label(bars)

plt.show()

## Bowler Analysis 2: Economy Rate

Only consider bowlers who have bowled enough deliveries.

In [ ]:
bowler_stats = (
    ipl_dataset.groupby('bowler')
    .agg({
        'runs_bowler':'sum',
        'valid_ball':'sum'
    })
)

bowler_stats = bowler_stats[
    bowler_stats['valid_ball'] >= 500
]

bowler_stats['Overs'] = (
    bowler_stats['valid_ball'] / 6
)

bowler_stats['Economy'] = (
    bowler_stats['runs_bowler']
    /
    bowler_stats['Overs']
)

In [ ]:
best_economy = (
    bowler_stats['Economy']
    .sort_values()
    .head(10)
)

best_economy

## Bowler Analysis 3: Player of the Match Impact

In [ ]:
potm_bowlers = (
    matches['player_of_match']
    .value_counts()
)

## Bowler Analysis 4: Death Over Specialists

In [ ]:
#Wickets in Death Overs
death_wickets = (
    ipl_dataset[
        (ipl_dataset['over'] >= 16)
        &
        (ipl_dataset['player_out'].notna())
    ]
    .groupby('bowler')
    .size()
    .sort_values(ascending=False)
)

death_wickets.head(10)

## Bowler Analysis 5: Longevity

In [ ]:
bowler_seasons = (
    ipl_dataset.groupby('bowler')['season']
    .nunique()
)

## Bowler Reliability Index (BoRI)

In [ ]:
bori = pd.DataFrame({
    'Wickets': top_wickets,
    'Economy': bowler_stats['Economy'],
    'Death_Wickets': death_wickets,
    'Seasons': bowler_seasons,
    'POTM': potm_bowlers
})

In [ ]:
bori = bori.fillna(0)

In [ ]:
#Normalize
bori['Wicket_Norm'] = (
    bori['Wickets']
    /
    bori['Wickets'].max()
)

bori['Economy_Norm'] = (
    1 - (
        bori['Economy']
        /
        bori['Economy'].max()
    )
)

bori['Death_Norm'] = (
    bori['Death_Wickets']
    /
    bori['Death_Wickets'].max()
)

bori['Season_Norm'] = (
    bori['Seasons']
    /
    bori['Seasons'].max()
)

bori['POTM_Norm'] = (
    bori['POTM']
    /
    bori['POTM'].max()
)

In [ ]:
#Reliability
bori['BoRI'] = (
    0.40 * bori['Wicket_Norm']
    + 0.25 * bori['Economy_Norm']
    + 0.15 * bori['POTM_Norm']
    + 0.10 * bori['Death_Norm']
    + 0.10 * bori['Season_Norm']
)

In [ ]:
#Top auction targets
top_bowlers = (
    bori.sort_values(
        'BoRI',
        ascending=False
    )
    .head(10)
)

top_bowlers

In [ ]:
plt.figure(figsize=(10,5))

bars = plt.bar(
    top_bowlers.index,
    top_bowlers['BoRI']
)

plt.title("Top Bowlers by Reliability Index")
plt.ylabel("BoRI Score")

plt.xticks(rotation=45)

plt.bar_label(bars, fmt='%.2f')

plt.show()

## All-Rounder Analysis

Step 1: Identify All-Rounders

Players who have:

Faced at least 300 balls
Bowled at least 300 valid deliveries

In [ ]:
batters = (
    ipl_dataset.groupby('batter')
    .agg({
        'runs_batter':'sum',
        'balls_faced':'sum'
    })
)

bowlers = (
    ipl_dataset.groupby('bowler')
    .agg({
        'valid_ball':'sum',
        'runs_bowler':'sum'
    })
)

all_rounders = (
    batters.merge(
        bowlers,
        left_index=True,
        right_index=True,
        how='inner'
    )
)

all_rounders = all_rounders[
    (all_rounders['balls_faced'] >= 300)
    &
    (all_rounders['valid_ball'] >= 300)
]

In [ ]:
#Step 2: Batting Strike Rate
all_rounders['Strike_Rate'] = (
    all_rounders['runs_batter']
    /
    all_rounders['balls_faced']
)*100

In [ ]:
#Step 3: Wickets
wickets = (
    ipl_dataset[
        ipl_dataset['player_out'].notna()
    ]
    .groupby('bowler')
    .size()
)

all_rounders['Wickets'] = wickets

In [ ]:
#Step 4: Economy
all_rounders['Overs'] = (
    all_rounders['valid_ball']/6
)

all_rounders['Economy'] = (
    all_rounders['runs_bowler']
    /
    all_rounders['Overs']
)

In [ ]:
#Step 5: POTM Awards
potm = matches['player_of_match'].value_counts()

all_rounders['POTM'] = potm

In [ ]:
#Step 6: Seasons Played
seasons_bat = (
    ipl_dataset.groupby('batter')['season']
    .nunique()
)

all_rounders['Seasons'] = seasons_bat

In [ ]:
#Fill Missing Values
all_rounders = all_rounders.fillna(0)

In [ ]:
#Normalize
all_rounders['Runs_Norm'] = (
    all_rounders['runs_batter']
    /
    all_rounders['runs_batter'].max()
)

all_rounders['SR_Norm'] = (
    all_rounders['Strike_Rate']
    /
    all_rounders['Strike_Rate'].max()
)

all_rounders['Wicket_Norm'] = (
    all_rounders['Wickets']
    /
    all_rounders['Wickets'].max()
)

all_rounders['Economy_Norm'] = (
    1 -
    all_rounders['Economy']
    /
    all_rounders['Economy'].max()
)

all_rounders['POTM_Norm'] = (
    all_rounders['POTM']
    /
    all_rounders['POTM'].max()
)

all_rounders['Season_Norm'] = (
    all_rounders['Seasons']
    /
    all_rounders['Seasons'].max()
)

## All-Rounder Reliability Index (ARI)

Weights:

Metric	        Weight

Runs	         20%

Strike Rate	     15%

Wickets	         25%

Economy	         20%

POTM	         10%

Seasons	         10%

In [ ]:
all_rounders['ARI'] = (
    0.20*all_rounders['Runs_Norm']
    +
    0.15*all_rounders['SR_Norm']
    +
    0.25*all_rounders['Wicket_Norm']
    +
    0.20*all_rounders['Economy_Norm']
    +
    0.10*all_rounders['POTM_Norm']
    +
    0.10*all_rounders['Season_Norm']
)

In [ ]:
#Top All-Rounders
top_all_rounders = (
    all_rounders
    .sort_values(
        'ARI',
        ascending=False
    )
    .head(10)
)

top_all_rounders

In [ ]:
plt.figure(figsize=(10,5))

bars = plt.bar(
    top_all_rounders.index,
    top_all_rounders['ARI']
)

plt.title("Top All-Rounders by Reliability Index")
plt.ylabel("ARI Score")

plt.xticks(rotation=45)

plt.bar_label(bars, fmt='%.2f')

plt.show()

# Auction Recommendations

In [ ]:
top_batters[['Runs','Strike_Rate','POTM','Death_Runs','Seasons','BRI']].head()

These batters combine consistency, scoring efficiency, finishing ability, and match-winning impact, making them the safest auction investments.

In [ ]:
top_bowlers[['Wickets','Economy','Death_Wickets','Seasons','POTM','BoRI']].head()

These bowlers consistently take wickets, maintain economical spells, and perform under pressure, making them valuable long-term assets.

In [ ]:
top_all_rounders[
    ['runs_batter',
     'Strike_Rate',
     'Wickets',
     'Economy',
     'POTM',
     'ARI']
].head(5)

In [ ]:
plt.figure(figsize=(10,5))

bars = plt.bar(
    top_all_rounders.index[:5],
    top_all_rounders['ARI'][:5]
)

plt.title("Top All-Rounders by ARI")
plt.ylabel("ARI Score")

plt.xticks(rotation=45)

plt.bar_label(bars, fmt='%.2f')

plt.show()

In [ ]:
death_runs.sort_values(ascending=False).head()

These players excel in the death overs, where matches are often decided.

In [ ]:
best_finishers = (
    death_runs
    .sort_values(ascending=False)
    .head(10)
)

best_finishers

In [ ]:
plt.figure(figsize=(10,5))

bars = plt.bar(
    best_finishers.index,
    best_finishers.values
)

plt.title("Best Finishers in IPL")
plt.ylabel("Death Over Runs")

plt.xticks(rotation=45)

plt.bar_label(bars)

plt.show()

In [ ]:
death_wickets.sort_values(ascending=False).head()


Death-over specialists can defend totals and break partnerships in crucial moments.

In [ ]:
best_death_bowlers = (
    death_wickets
    .sort_values(ascending=False)
    .head(10)
)

best_death_bowlers

In [ ]:
matches['player_of_match'].value_counts().head(10)

Player of the Match awards reflect a player's ability to influence outcomes rather than simply accumulate statistics.

In [ ]:
auction_recommendations = pd.DataFrame({
    'Top_Batters': top_batters.index[:5],
    'Top_Bowlers': top_bowlers.index[:5]
})

auction_recommendations

In [ ]:
# Safest Auction Investments

auction_summary = pd.DataFrame({
    'Category': [
        'Best Batter Investment',
        'Best Bowler Investment',
        'Best All-Rounder Investment',
        'Best Finisher',
        'Best Death Bowler'
    ],

    'Recommended Player': [
        top_batters.index[0],
        top_bowlers.index[0],
        top_all_rounders.index[0],
        best_finishers.index[0],
        best_death_bowlers.index[0]
    ]
})

print(auction_summary)

# Final Conclusion

This analysis demonstrates that successful IPL auction strategies should be based on data-driven insights rather than reputation or recent performances alone. Historical IPL data was analyzed to understand team success, venue influence, batting performance, bowling performance, and player reliability.

### Key Findings

* Mumbai Indians and Chennai Super Kings have been the most successful franchises, highlighting the importance of long-term squad stability.
* Toss advantage has only a limited impact on match outcomes, emphasizing the importance of team quality over luck.
* Venue conditions influence player performances and should be considered while evaluating players.
* Total runs and wickets alone are insufficient to measure a player's true value.
* Consistency, strike rate, economy rate, death-over performance, match-winning ability, and longevity are critical indicators of player reliability.
* Separate reliability indices were developed for batters (BRI), bowlers (BoRI), and all-rounders (ARI) to ensure fair and role-specific evaluation.
* Specialized roles such as finishers and death bowlers were also identified, as these players often decide the outcome of close matches.
* Data-driven evaluation helps franchises maximize returns on investment while minimizing auction risk.

### Final Recommendation

Championship-winning teams are not built by investing in the most popular players, but by investing in the most reliable performers. By combining performance, consistency, impact, and longevity, the Batter Reliability Index (BRI), Bowler Reliability Index (BoRI), and All-Rounder Reliability Index (ARI) provide a comprehensive framework for identifying the safest and most valuable auction investments for IPL franchises.
